# 数据并行
在数据并行系统中，每个计算设备都有整个神经网络模型的完整副本（Model Replica），进行迭代时，每个计算设备只分配了一个批次数据样本的子集，并根据该批次样本子集的数据进行网络模型的前向计算。假设一个批次的训练样本数为 $N$ ，使用 $M$ 个计算设备并行计算，每个计算设备会分配到 $N/M$ 个样本。前向计算完成后，每个计算设备都会根据本地样本计算损失误差得到梯度 $G_{i}$ （i 为加速卡编号），并将本地梯度 $G_{i}$ 进行广播。所有计算设备需要聚合其他加速度卡给出的梯度值，然后使用平均梯度 $(\Sigma_{i=1}^{N}G_{i})/N$ 对模型进行更新，完成该批次训练。图4.4给出了由两个计算设备组成的数据并行训练系统样例。  

![](images/47d19f5f99aec8d27084548083e2bc46dbb4c0cdfe31b0d2f3b5fc52fccb9f7a.jpg)  

数据并行训练系统可以通过增加计算设备，有效提升整体训练吞吐量，每秒全局批次数（GlobalBatch Size Per Second) 。它和单计算设备训练相比，最主要的区别就在于反向计算中的梯度需要在所有计算设备中进行同步，以保证每个计算设备上最终得到的是所有进程上梯度的平均值。常见的神经网络框架中都有数据并行方式的具体实现，包括：TensorFlow DistributedStrategy、PyTorchDistributed、Horovod DistributedOptimizer 等。由于基于Transformer 架构的大语言模型中每个算子都是依赖单个数据而非批次数据，因此数据并行并不会影响其计算逻辑，一般情况下各训练设备中前向计算是独立的，不涉及同步问题。数据并行训练加速比最高，但要求每个设备上都备份一份模型，显存占用比较高。  

使用PyTorch `DistributedDataParallel` 实现单个服务器多加速卡训练代码如下，首先构造`Dis-tributedSampler` 类，将数据集的样本随机打乱并分配到不同计算设备：  

In [ ]:
import argparse
import os
import shutil
import time
import warnings
import numpy as np
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.distributed as dist
import torch.optim
import torch.utils.data
import torch.utils.data.distributed
from torch.utils.data.distributed import DistributedSampler
from models import DeepLab
from dataset import Cityscaples

In [ ]:
class DistributedSampler(Sampler):
    def __init__(self, dataset, num_replicas=None, rank=None, shuffle=True, seed=0):
        if num_replicas is None:
            if not dist.is_available():
                raise RuntimeError("Requires distributed package to be available")
            num_replicas = dist.get_world_size()
        if rank is None:
            if not dist.is_available():
                raise RuntimeError("Requires distributed package to be available")
            rank = dist.get_rank()
        self.dataset = dataset # 数据集
        self.num_replicas = num_replicas # 进程个数 默认等于 world_size(GPU 个数)
        self.rank = rank # 当前属于哪个进程/哪块 GPU
        self.epoch = 0
        self.num_samples = int(math.ceil(len(self.dataset) * 1.0 / self.num_replicas))
        # 每个进程的样本个数
        self.total_size = self.num_samples * self.num_replicas # 数据集总样本的个数
        self.shuffle = shuffle # 是否要打乱数据集
        self.seed = seed # 种子
        
    def __iter__(self):
    # 1、 Shuffle 处理：打乱数据集顺序
        if self.shuffle:
            # 根据 epoch 和种子进行混淆
            g = torch.Generator()
            # 这里 self.seed 是一个定值，通过 set_epoch 改变 self.epoch 可以改变我们的初始化种子
            # 这就可以让每一个 epoch 中数据集的打乱顺序不同，使每一个 epoch 中，
            # 每一块 GPU 拿到的数据都不一样，这样可以有利于更好的训练
            g.manual_seed(self.seed + self.epoch)
            indices = torch.randperm(len(self.dataset), generator=g).tolist()
        else:
            indices = list(range(len(self.dataset)))
        # 数据补充
        indices += indices[:(self.total_size- len(indices))]
        assert len(indices) == self.total_size
        # 分配数据
        indices = indices[self.rank:self.total_size:self.num_replicas]
        assert len(indices) == self.num_samples
        return iter(indices)

    def __len__(self):
        return self.num_samples
    
    def set_epoch(self, epoch):
        r"""
        设置此采样器的训练轮数 epoch。
        当 :attr:`shuffle=True` 时，确保所有副本在每个轮数使用不同的随机顺序。
        否则，此采样器的下一次迭代将产生相同的顺序。
        Arguments:
        epoch (int): 训练轮数
        """
        self.epoch = epoch


利用DistributedSampler 构造完整的训练程序样例main.py如下：

In [ ]:

# 参数设置
parser = argparse.ArgumentParser(description='DeepLab')
parser.add_argument('-j', '--workers', default=4, type=int, metavar='N',
help='number of data loading workers (default: 4)')
parser.add_argument('--epochs', default=100, type=int, metavar='N',
help='number of total epochs to run')
parser.add_argument('--start-epoch', default=0, type=int, metavar='N',
help='manual epoch number (useful on restarts)')
parser.add_argument('-b', '--batch-size', default=3, type=int,
metavar='N')
parser.add_argument('--local_rank', default=0, type=int, help='node rank for distributed training')
args = parser.parse_args()
torch.distributed.init_process_group(backend="nccl") # 初始化
print("Use GPU: {} for training".format(args.local_rank))
# 创建模型
model = DeepLab()
torch.cuda.set_device(args.local_rank) # 当前显卡
model = model.cuda() # 模型放置于显卡上
model = torch.nn.parallel.DistributedDataParallel(model, device_ids=[args.local_rank],
    output_device=args.local_rank, find_unused_parameters=True) # 数据并行
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), args.lr,
    momentum=args.momentum, weight_decay=args.weight_decay)
train_dataset = Cityscaples()
train_sampler = DistributedSampler(train_dataset) # 分配数据
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=args.batch_size,
    shuffle=False, num_workers=args.workers, pin_memory=True, sampler=train_sampler)

通过以下命令行启动上述程序：
```bash
CUDA_VISIBLE_DEVICES=0,1 python-m torch.distributed.launch--nproc_per_node=2 main.py
```